In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as st

In [2]:
def cohen_glass_hedges(x,y):
    nx = len(x)
    ny = len(y)
    pstdnmr = (nx-1)*x.var(ddof=1) + (ny-1)*y.var(ddof=1)   # (n-1)(var1+var2)
    pstddnm = nx + ny - 2                                   # (n-1)*2
    pstd = (pstdnmr / pstddnm)**(0.5)
    cohen = (x.mean() - y.mean()) / pstd
    glass = (x.mean() - y.mean()) / x.std(ddof=1)
    hedges = cohen * (1 - 3/(4 * (nx+ny) - 9) )             # cohen * (1-3/(8*n-9))
    return (cohen, glass, hedges)

In [3]:
o = pd.read_excel('OptoJumpSorted.xlsx', sheet_name='GeneralSbs')
ost = pd.read_excel('OptoJumpSorted.xlsx', sheet_name='GeneralSt')

In [4]:
descr = o.loc[:,'WPX_1J1':].agg(['count', 'sum', 'median', 'mean', 'min', 'max', 'std', 'var']).round(2).T
descr.loc[:,'sum'] = 39

In [5]:
descr['kurtosis'] = o.loc[:,'WPX_1J1':].apply(st.kurtosis).round(2)
descr['skew'] = o.loc[:,'WPX_1J1':].apply(st.skew).round(2)
descr[['shapiro statistic', 'shapiro p value']] = o.loc[:,'WPX_1J1':].apply(st.shapiro).round(2).T

In [6]:
descrt = descr.T
replacements = {"1": "_final", "2": "_initial"}
descrt = descrt.rename(columns=lambda col: col[:-1] + replacements[col[-1]] if col[-1] in replacements else col)
descrt.columns = descrt.columns.str.replace('_',' ')
descr = descrt.T

In [7]:
st.t.ppf(0.025,39)

np.float64(-2.022690920036761)

In [8]:
ost.loc[:40, 'Test'] = 'final'
ost.loc[40:, 'Test'] = 'initial'

In [9]:
ost.columns = ['Index', 'Name', 'Test', 'Group', 'Sex', 'Weight', 'Height', 'WPX 1 Jump',
       'Step Width 1 Jump 1', 'Step Width 1 Jump 2', 'Flight Time 1 Jump', 'Elevation 1 Jump', 'Jumps',
       'Flight Time', 'Contact Time', 'Elevation', 'Propulsive Power', 'Pace', 'RSI', 'WPX',
       'WPY', 'Step Width', 'Distance', 'Total Time', 'Jump Time', 'Slope - Flight Time',
       'Slope - Contact Time', 'Slope - Elevation', 'Slope - Power', 'Slope - Pace', 'Slope - RSI',
       'Slope - Step Width', 'Slope - Distance', 'Flight Time BFI', 'Contact Time BFI',
       'Elevation BFI', 'Power BFI', 'Pace BFI', 'RSI BFI', 'Step Width BFI', 'Distance BFI']

In [10]:
tt = pd.DataFrame(data=0.1, index=ost.loc[:,'WPX 1 Jump':].columns,
                  columns=['Significance Level', 'df',
                           'T critical (two tails)', 'T statistic','T test p-value',
                           'Levene statistic', 'Levene p-value', 'Effect size (Cohen d)'])

In [11]:
tt.loc[:,'Significance Level'] = 0.05
tt.loc[:,'df'] = 39
tt.loc[:,'T critical (two tails)'] = 2.02

In [12]:
for col in ost.loc[:,'WPX 1 Jump':].columns:
    tt.loc[col,'T statistic'] = st.ttest_rel(ost.loc[40:,col],ost.loc[:39,col])[0]
    tt.loc[col,'T test p-value'] = st.ttest_rel(ost.loc[40:,col],ost.loc[:39,col])[1]
    tt.loc[col,'Levene statistic'] = st.levene(ost.loc[40:,col],ost.loc[:39,col])[0]
    tt.loc[col,'Levene p-value'] = st.levene(ost.loc[40:,col],ost.loc[:39,col])[1]
    tt.loc[col,'Effect size (Cohen d)'] = cohen_glass_hedges(ost.loc[40:,col],ost.loc[:39,col])[0]


In [13]:
tt = tt.round(2)

In [14]:
tt

,Significance Level,df,T critical (two tails),T statistic,T test p-value,Levene statistic,Levene p-value,Effect size (Cohen d)
WPX 1 Jump,0.05,39.0,2.02,-1.14,0.26,4.28,0.04,-0.27
Step Width 1 Jump 1,0.05,39.0,2.02,2.70,0.01,1.34,0.25,0.59
Step Width 1 Jump 2,0.05,39.0,2.02,0.07,0.94,0.30,0.58,0.02
Flight Time 1 Jump,0.05,39.0,2.02,1.21,0.23,0.52,0.47,0.22
Elevation 1 Jump,0.05,39.0,2.02,1.13,0.27,0.18,0.67,0.22
Jumps,0.05,39.0,2.02,1.27,0.21,0.03,0.87,0.30
Flight Time,0.05,39.0,2.02,-2.69,0.01,0.01,0.94,-0.56
Contact Time,0.05,39.0,2.02,2.06,0.05,1.23,0.27,0.34
Elevation,0.05,39.0,2.02,-2.46,0.02,0.54,0.47,-0.55
Propulsive Power,0.05,39.0,2.02,-2.53,0.02,0.13,0.72,-0.54


In [15]:
descr

,count,sum,median,mean,min,max,std,var,kurtosis,skew,shapiro statistic,shapiro p value
WPX 1J final,40.0,39.0,1.05,-0.53,-30.80,22.4,10.37,107.47,0.94,-0.64,0.96,0.23
WPX 1J initial,40.0,39.0,-2.35,-2.88,-17.20,16.6,6.91,47.77,0.58,0.16,0.97,0.33
Step Width 1J 1 final,40.0,39.0,25.00,24.53,0.00,32.3,5.16,26.67,12.12,-3.17,0.64,0.00
Step Width 1J 1 initial,40.0,39.0,27.10,26.88,20.80,32.3,2.32,5.40,0.12,-0.17,0.97,0.39
Step Width 1J 2 final,40.0,39.0,28.65,31.67,22.90,64.6,9.19,84.46,3.81,2.01,0.75,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
Bosco RSI initial,40.0,39.0,90.15,89.07,65.78,100.0,9.41,88.59,-0.27,-0.73,0.92,0.01
Bosco Step Width final,40.0,39.0,65.28,66.65,31.35,100.0,15.09,227.63,0.76,0.56,0.93,0.01
Bosco Step Width initial,40.0,39.0,63.63,68.04,27.47,100.0,20.38,415.36,-0.52,0.13,0.92,0.01
Bosco Distance final,40.0,39.0,97.43,97.45,91.65,100.0,1.80,3.25,1.40,-0.88,0.93,0.02


In [ ]:
descrt.to_csv('descriptivet.csv', index=False)
descr.to_csv('descriptive.csv', index=False)
tt.to_csv('tt.csv', index=False)
tt.T.to_csv('ttt.csv', index=False)